[Reference](https://medium.com/codetodeploy/getting-started-with-langgraph-build-your-first-chatbot-in-15-minutes-234f1792cbb7)

```
- A node (a function)
- Connected by edges
- Sharing a state (memory)
```

# Step 1: Install Dependencies


In [1]:
!pip install langgraph langchain langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 4.4 MB/s eta 0:00:00


```
export OPENAI_API_KEY="your_api_key"
```

# Step 2: Define the Chat State

In [2]:
from typing import TypedDict, List
from langchain_core.messages import BaseMessage
class ChatState(TypedDict):
    messages: List[BaseMessage]

# Step 3: Initialize the LLM

In [3]:
# If you are using windows, then use python-dotenv to load env variables
from dotenv import load_dotenv
load_dotenv()
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7
)

# Step 4: Create the Chatbot Node

In [4]:
def chatbot_node(state: ChatState):
    response = llm.invoke(state["messages"])
    return {
        "messages": state["messages"] + [response]
    }

# Step 5: Build the LangGraph

In [5]:
from langgraph.graph import StateGraph
graph = StateGraph(ChatState)
graph.add_node("chatbot", chatbot_node)
graph.set_entry_point("chatbot")
graph.set_finish_point("chatbot")
app = graph.compile()

# Step 6: Run the Chatbot in a Loop (Jupyter-Friendly)

In [6]:
from langchain_core.messages import HumanMessage
state = {"messages": []}
while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        print("Goodbye 👋")
        break
    state["messages"].append(HumanMessage(content=user_input))
    state = app.invoke(state)
    print("Bot:", state["messages"][-1].content)